# Import libraries

## Biopython gives us access to PubMed's Entrez API

In [ ]:
### Bio = Biopython library
### Entrez = PubMed's API tool inside it

from Bio import Entrez
import json

# Always tell PubMed who you are (required by their API)
Entrez.email = "tpriya27@gmail.com"

print("Libraries imported successfully!")


Libraries imported successfully!


## Block 1: Data Ingestion 

### Search PubMed for Alzheimer's papers
Entrez.efetch() to retrieve full bibliographic information or abstracts. This automated approach is especially valuable when you need to analyze hundreds or thousands of papers programmatically.



In [19]:
# We're asking PubMed: "give me paper IDs 
# related to Alzheimer's early detection"
#db="pubmed"     → which database to search
#term="..."      → your search query
#retmax=10       → maximum 10 results
#paper_ids       → list of PubMed ID numbers
                  

search_handle = Entrez.esearch(
    db="pubmed",
    term="Alzheimer's early detection biomarkers",
    retmax=10
)

search_results = Entrez.read(search_handle)
search_handle.close()

paper_ids = search_results["IdList"]

print("Found paper IDs:", paper_ids)
print("Total found:", len(paper_ids))

Found paper IDs: ['41773883', '41773433', '41772333', '41771832', '41770931', '41768934', '41767159', '41767157', '41764603', '41763632']
Total found: 10


## Fetch The Actual Papers

####  IDs to get the real titles and abstracts:

In [20]:
fetch_handle = Entrez.efetch(
    db="pubmed",
    id=paper_ids,
    rettype="abstract",
    retmode="xml"
)

papers = Entrez.read(fetch_handle)
fetch_handle.close()

print("Papers fetched successfully!")
print("Number of papers:", len(papers["PubmedArticle"]))

Papers fetched successfully!
Number of papers: 10


## Extract The Useful Information

####  This is the data we'll later convert to embeddings

In [21]:
#articles        → list of all 10 papers
#enumerate()     → gives index + item together
#try/except      → some papers have no abstract
                  #we handle that gracefully
#[:200]          → show only first 200 characters
                  #abstracts are very long!

In [22]:


articles = papers["PubmedArticle"]

for i, article in enumerate(articles):
    
    # Extract title
    title = article["MedlineCitation"]["Article"]["ArticleTitle"]
    
    # Extract abstract (some papers may not have one)
    try:
        abstract = article["MedlineCitation"]["Article"]["Abstract"]["AbstractText"][0]
    except KeyError:
        abstract = "No abstract available"
    
    print(f"Paper {i+1}:")
    print(f"Title: {title}")
    print(f"Abstract: {str(abstract)[:200]}...")
    print("-" * 50)

Paper 1:
Title: Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease.
Abstract: MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis....
--------------------------------------------------
Paper 2:
Title: Plasma p-tau217, p-tau181 and Aβ42/40 for Alzheimer's disease diagnosis: ROC accuracy and 18F-florbetapir amyloid PET-CT concordance.
Abstract: Early diagnosis of Alzheimer's disease (AD) presents significant challenges. This study assessed the diagnostic utility of seven plasma biomarkers and PET-CT imaging in cognitively healthy individuals...
--------------------------------------------------
Paper 3:
Title: Progression of fiber bundle damage in amnestic Alzheimer's disease and LATE: a 2-year fixel-based study.
Abstract: Alzheimer's disease (AD) and limbic-predominant age-related TDP-43 encephalopathy (LATE) are two neurodegenerative diseases underpinned by distinct protei

Real, cutting-edge Alzheimer's research papers fetched live from PubMed! 
- ✅ "Tears as a window to Alzheimer's disease: biomarkers for early detection"
- ✅ "Early Diagnosis of Alzheimer's: Machine Learning Analysis"
- ✅ "Plasma GFAP and P-tau217 with imaging ATN markers"
These are 2026 papers — fresher than any AI's training data!

## Save The Data 

In [23]:
# Cell 5: Structure and save the data as JSON
# This becomes our raw data for the RAG pipeline

import json

structured_papers = []

for article in articles:
    
    title = article["MedlineCitation"]["Article"]["ArticleTitle"]
    
    try:
        abstract = article["MedlineCitation"]["Article"]["Abstract"]["AbstractText"][0]
    except KeyError:
        abstract = "No abstract available"
    
    # Extract PubMed ID
    pmid = str(article["MedlineCitation"]["PMID"])
    
    structured_papers.append({
        "pmid": pmid,
        "title": str(title),
        "abstract": str(abstract),
        "source": "PubMed"
    })

# Save to data folder
with open("../data/raw/alzheimer_papers.json", "w") as f:   
    json.dump(structured_papers, f, indent=2)

print(f"Saved {len(structured_papers)} papers to alzheimer_papers.json")
print("\nSample paper:")
print(json.dumps(structured_papers[0], indent=2))

Saved 10 papers to alzheimer_papers.json

Sample paper:
{
  "pmid": "41773883",
  "title": "Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease.",
  "abstract": "MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis.",
  "source": "PubMed"
}
